In [1]:
import requests

AGENTS = [
    "http://localhost:8000",  # math-agent
    "http://localhost:8001"   # finance-agent
]

def discover_capabilities():
    registry = []

    for url in AGENTS:
        info = requests.get(f"{url}/agent-info").json()
        tools = requests.get(f"{url}/tools").json()

        registry.append({
            "agent": info["name"],
            "description": info["description"],
            "endpoint": info["endpoint"],
            "tools": tools["tools"]
        })

    return registry

In [2]:
import requests

AGENTS = [
    "http://localhost:8000",  # math-agent
    "http://localhost:8001"   # finance-agent
]

def discover_capabilities():
    registry = []

    for url in AGENTS:
        info = requests.get(f"{url}/agent-info").json()
        tools = requests.get(f"{url}/tools").json()

        registry.append({
            "agent": info["name"],
            "description": info["description"],
            "endpoint": info["endpoint"],
            "tools": tools["tools"]
        })

    return registry

In [10]:
def route(query, registry):
    query = query.lower()

    for agent in registry:
        for tool in agent["tools"]:

            tool_name = tool["name"].lower().replace("_", " ")
            tool_desc = tool["description"].lower()

            # match against:
            # 1. tool name (with spaces)
            # 2. tool description words
            if tool_name in query or any(word in query for word in tool_desc.split()):
                return agent

    return None

## Introducing MCP for Execution 

In [11]:
import re

def build_payload(query, agent):
    numbers = list(map(float, re.findall(r"\d+\.?\d*", query)))

    tool_name = agent["tools"][0]["name"]

    if tool_name == "calculate_interest":
        return {
            "tool": "calculate_interest",
            "args": {
                "principal": numbers[0],
                "rate": numbers[1],
                "time": numbers[2]
            }
        }

    elif tool_name == "add":
        return {
            "tool": "add",
            "args": {
                "a": numbers[0],
                "b": numbers[1]
            }
        }

    return None

In [12]:
import re

def build_payload(query, agent):
    numbers = list(map(float, re.findall(r"\d+\.?\d*", query)))

    tool_name = agent["tools"][0]["name"]

    if tool_name == "calculate_interest":
        return {
            "tool": "calculate_interest",
            "args": {
                "principal": numbers[0],
                "rate": numbers[1],
                "time": numbers[2]
            }
        }

    elif tool_name == "add":
        return {
            "tool": "add",
            "args": {
                "a": numbers[0],
                "b": numbers[1]
            }
        }

    return None

In [13]:
def invoke(agent, payload):
    response = requests.post(agent["endpoint"], json=payload)
    return response.json()

In [14]:
def execute(query):
    print("\n🔍 QUERY:", query)

    # Step 1: Discover
    registry = discover_capabilities()

    # Step 2: Route
    agent = route(query, registry)

    if not agent:
        return "❌ No agent found"

    print("🧠 SELECTED AGENT:", agent["agent"])

    # Step 3: Build Payload
    payload = build_payload(query, agent)

    print("📦 PAYLOAD:", payload)

    # Step 4: Invoke
    result = invoke(agent, payload)

    print("✅ RESULT:", result)

    return result

In [15]:
query1 = "calculate interest for 1000 at 5 for 2 years"
query2 = "add 10 and 20"

execute(query1)
execute(query2)


🔍 QUERY: calculate interest for 1000 at 5 for 2 years
🧠 SELECTED AGENT: finance-agent
📦 PAYLOAD: {'tool': 'calculate_interest', 'args': {'principal': 1000.0, 'rate': 5.0, 'time': 2.0}}
✅ RESULT: {'status': 'success', 'result': 100.0}

🔍 QUERY: add 10 and 20
🧠 SELECTED AGENT: math-agent
📦 PAYLOAD: {'tool': 'add', 'args': {'a': 10.0, 'b': 20.0}}
✅ RESULT: {'status': 'success', 'result': 30.0}


{'status': 'success', 'result': 30.0}